In [4]:
# 1.2
# 1. import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

# 2. import csv
data = pd.read_csv('./data_and_materials/gamma-ray.csv')
# check the data
data.head()


,seconds,count
0,116.0,0.0
1,112.0,0.0
2,160.0,0.0
3,51.5,0.0
4,102.0,1.0


## Problem 1.2, part (d)

Under the null model, the emission rate is constant:

$$G_i \sim \mathrm{Poisson}(\lambda t_i).$$

The likelihood is

$$L(\lambda)=\prod_i \frac{e^{-\lambda t_i}(\lambda t_i)^{G_i}}{G_i!}.$$

The log-likelihood, up to constants that do not depend on $\lambda$, is

$$\ell(\lambda)=-\lambda\sum_i t_i + \left(\sum_i G_i\right)\log \lambda + C.$$

Differentiate and set the derivative equal to zero:

$$\frac{d\ell}{d\lambda}=-\sum_i t_i + \frac{\sum_i G_i}{\lambda}=0.$$

Therefore the MLE is

$$\hat\lambda=\frac{\sum_i G_i}{\sum_i t_i}.$$


In [ ]:
total_count = data['count'].sum()
total_time = data['seconds'].sum()
lambda_hat_null = total_count / total_time

print(f"sum G_i = {total_count:g}")
print(f"sum t_i = {total_time:g}")
print(f"lambda_hat = {lambda_hat_null}")
print(f"3 significant figures: {lambda_hat_null:.3g}")

For the answer box in part (d), enter

$$\boxed{0.00388}$$

This is a rate per second. It is not the total number of observed gamma rays.

## Problem 1.2, part (e)

Under the alternative model, each interval has its own rate:

$$G_i \sim \mathrm{Poisson}(\lambda_i t_i).$$

For one interval, the log-likelihood is

$$\ell_i(\lambda_i)=-\lambda_i t_i + G_i\log(\lambda_i t_i)-\log(G_i!).$$

Keeping only terms involving $\lambda_i$:

$$\ell_i(\lambda_i)=-\lambda_i t_i + G_i\log\lambda_i + C.$$

Differentiate and set equal to zero:

$$\frac{d\ell_i}{d\lambda_i}=-t_i+\frac{G_i}{\lambda_i}=0.$$

So the MLE for each interval is

$$\hat\lambda_i=\frac{G_i}{t_i}.$$

For the answer box in part (e), enter `G_i/t_i`.

In [ ]:
data['lambda_hat_i'] = data['count'] / data['seconds']
data.head()

## Problem 1.2, part (f)

We use the likelihood ratio test statistic

$$\Lambda(x)=-2\log\left(\frac{\max_{\lambda} f(G_0,\ldots,G_{99}\mid \lambda)}{\max_{\lambda_0,\ldots,\lambda_{99}} f(G_0,\ldots,G_{99}\mid \lambda_0,\ldots,\lambda_{99})}\right).$$

This compares the best likelihood under the null model with the best likelihood under the alternative model. Large values mean the alternative model fits much better than the null model, so large values give evidence against the null hypothesis.

### Part (f-2)

The null model has 1 parameter, $\lambda$. The alternative model has 100 parameters, $\lambda_0,\ldots,\lambda_{99}$.

By Wilks' theorem, the asymptotic distribution of the likelihood ratio statistic is

$$\chi^2_{100-1}=\chi^2_{99}.$$

So the correct choice is $\chi^2_{99}$.

### Parts (f-3) and (f-4)

At significance level 0.05, the rejection region is the upper 5% tail of $\chi^2_{99}$:

$$\Lambda(x) \geq \chi^2_{99,0.95}.$$

The observed statistic is

$$\Lambda(x)=2\left(\ell_{\mathrm{alt}}(\hat\lambda_0,\ldots,\hat\lambda_{99})-\ell_{\mathrm{null}}(\hat\lambda)\right).$$

In [ ]:
from scipy.special import gammaln
from scipy.stats import chi2

g = data['count'].to_numpy()
t = data['seconds'].to_numpy()

# Null model: mu_i = lambda_hat * t_i
mu_null = lambda_hat_null * t
loglik_null = np.sum(g * np.log(mu_null) - mu_null - gammaln(g + 1))

# Alternative model: mu_i = lambda_hat_i * t_i = G_i.
# If G_i = 0, the maximized contribution is log P(G_i=0 | mu_i=0) = 0.
loglik_alt_terms = np.zeros_like(g, dtype=float)
positive = g > 0
loglik_alt_terms[positive] = (
    g[positive] * np.log(g[positive])
    - g[positive]
    - gammaln(g[positive] + 1)
)
loglik_alt = np.sum(loglik_alt_terms)

lrt_stat = 2 * (loglik_alt - loglik_null)
critical_value = chi2.ppf(0.95, df=99)
p_value = chi2.sf(lrt_stat, df=99)

print(f"loglik_null = {loglik_null}")
print(f"loglik_alt = {loglik_alt}")
print(f"Lambda(x) = {lrt_stat}")
print(f"critical value = {critical_value}")
print(f"p-value = {p_value}")
print()
print(f"3 significant figures for rejection threshold: {critical_value:.3g}")
print(f"3 significant figures for Lambda(x): {lrt_stat:.3g}")
print(f"3 significant figures for p-value: {p_value:.3g}")

For part (f-3), enter

$$\boxed{123}$$

For part (f-4), enter

$$\boxed{\Lambda(x)=104}$$

and

$$\boxed{p=0.336}$$

### Part (f-5)

Since $\Lambda(x)=104.398 < 123.225$, the statistic is not in the rejection region. Equivalently, the p-value is $0.336 > 0.05$.

Therefore, we cannot reject the null hypothesis. The data do not provide enough evidence that the emission rate is not constant.

Choose: **Cannot reject our null hypothesis, emission rate appear to be constant.**